# Laboratorio 06 — Publicar solo si cuadra

**Duración: 60 minutos.**

El laboratorio anterior cerró señalando algo que venía pasando desde el principio:

> **Todo lo que has hecho hasta ahora entró directo a la tabla que consulta todo el
> mundo.**
>
> Cuando cargaste el lote de 2026, Valentina podría haber estado consultando esa misma
> tabla en ese preciso momento. ¿Y si el lote nuevo viene malo?

Hoy resolvemos eso.

---

### Por qué le duele a la DGT

Un lote con errores que entra directo queda **visible para los analistas antes de que
nadie lo revise**. Y el error no se descubre en el momento: se descubre cuando alguien
ya reportó una cifra equivocada.

Corregir después no cuesta solo trabajo. Cuesta credibilidad.

Lo que falta es poder **cargar en un costado, revisar, y publicar solo si cuadra**.

### En la libreta

- Una **rama** es una libreta paralela, donde se anota lo que todavía no es oficial.
  Quien consulta la libreta que cuelga del clavo no la ve.
- **Publicar** es pasar lo revisado a la libreta oficial.
- Una **etiqueta** es ponerle nombre a una página, para volver a ella sin acordarse de
  la fecha: *"el cierre de junio"*.

---

### Cómo se usa este notebook

Como siempre. Dos avisos propios de hoy:

**Uno.** Vas a escribir **nombres de tabla a tres partes** más que nunca. Para escribir
en una rama se nombra `mi_espacio.recepcion_lab06.branch_<rama>`, con tu espacio adelante,
igual que las vistas del sistema.

**Dos.** Este laboratorio usa lo más nuevo de la versión de Iceberg que tiene tu
plataforma. Al final se dice qué **no** está disponible todavía en esta versión, porque
eso también hay que saberlo.


## Paso 0 — Preparar la tabla

Primero, a tu espacio de trabajo.

In [1]:
%%sql
-- Celda 0.1
USE mi_espacio

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/24 21:14:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Partimos de cero.

In [2]:
%%sql
-- Celda 0.2
DROP TABLE IF EXISTS recepcion_lab06

La tabla de recepción del mes, con un lote que ya está revisado y correcto. Esta
es la tabla que consultan los analistas de la DGT.

In [3]:
%%sql
-- Celda 0.3
CREATE TABLE recepcion_lab06 USING iceberg AS
SELECT * FROM curso.recepcion_limpia

¿Cuántos documentos y por cuánto?

In [4]:
%%sql
-- Celda 0.4
SELECT count(*)         AS documentos,
       sum(monto_total) AS total_del_mes
FROM recepcion_lab06

documentos,total_del_mes
1000,9117709433.18


## Paso 1 — El riesgo, mostrado

Mil documentos por algo más de nueve mil millones. Esa es la cifra que hoy sale
en los reportes.

Ahora llega el lote de recepción del día. Vamos a cargarlo **como lo hemos hecho todo
el curso**: directo a la tabla.

In [5]:
%%sql
-- Celda 1.1
INSERT INTO recepcion_lab06 SELECT * FROM curso.recepcion_sospechosa

Entró. Y ahora, como corresponde, pasemos los controles de calidad.

Son tres, y los tres son aritmética tributaria elemental:

| Columna | Cuántos documentos cuenta |
|---|---|
| `errores_iva` | Los que tienen el IVA distinto del 19 % del neto |
| `errores_suma` | Los que tienen el total distinto de neto + exento + IVA |
| `errores_negativos` | Los que tienen monto negativo sin ser nota de crédito (tipo 61) |

**Cómo cuenta cada columna.** Cada `CASE WHEN condición THEN 1 ELSE 0 END` mira un
documento y le pone un **1** si rompe la regla y un **0** si la cumple. El `sum` de
afuera suma esos unos, así que lo que sale no es un diagnóstico de la tabla: es la
**cantidad de documentos** que rompen esa regla.

Desarmemos la primera con un ejemplo. Un documento con neto 100.000 e IVA 25.000: el
19 % del neto es 19.000, la diferencia contra el IVA declarado es 6.000, el `abs` le
quita el signo, y 6.000 es mayor que 2. La regla se rompe, el documento aporta un 1, y
`errores_iva` sube en uno.

Los `> 2` son una tolerancia de dos pesos, por los redondeos.

In [6]:
%%sql
-- Celda 1.2
SELECT count(*) AS documentos,
       sum(CASE WHEN abs(monto_iva - round(monto_neto * 0.19)) > 2
                THEN 1 ELSE 0 END) AS errores_iva,
       sum(CASE WHEN abs(monto_total - (monto_neto + monto_exento + monto_iva)) > 2
                THEN 1 ELSE 0 END) AS errores_suma,
       sum(CASE WHEN monto_total < 0 AND tipo_dte <> 61
                THEN 1 ELSE 0 END) AS errores_negativos
FROM recepcion_lab06

documentos,errores_iva,errores_suma,errores_negativos
1500,13,20,7


**Ahí está el problema.** Hay documentos con el IVA descuadrado, con el total mal
sumado y con montos negativos que no deberían serlo.

Y fíjate en dónde están: **en la tabla**. Ya publicados. Si Valentina consulta ahora,
se lleva esas cifras.

Veamos cuántos documentos distintos fallan, porque un mismo documento puede fallar más
de un control a la vez.

In [7]:
%%sql
-- Celda 1.3
SELECT count(*) AS documentos_que_no_cuadran
FROM recepcion_lab06
WHERE abs(monto_iva - round(monto_neto * 0.19)) > 2
   OR abs(monto_total - (monto_neto + monto_exento + monto_iva)) > 2
   OR (monto_total < 0 AND tipo_dte <> 61)

documentos_que_no_cuadran
20


Veinte documentos malos, publicados.

Deshagámoslo con lo que ya sabes del laboratorio 01. Primero, las páginas.

In [8]:
%%sql
-- Celda 1.4
SELECT snapshot_id, committed_at, operation
FROM mi_espacio.recepcion_lab06.snapshots
ORDER BY committed_at

snapshot_id,committed_at,operation
3131769625158961713,2026-09-24 21:14:48.528,append
6361030606601007229,2026-09-24 21:14:50.106,append


Dos páginas: la carga limpia y la carga del lote sospechoso.

**Copia el `snapshot_id` de la primera** y vuelve a ella.

In [9]:
%%sql
-- Celda 1.5
CALL spark_catalog.system.rollback_to_snapshot('mi_espacio.recepcion_lab06', <TU_SNAPSHOT_ID>)

previous_snapshot_id,current_snapshot_id
6361030606601007229,3131769625158961713


Y comprobemos que la tabla quedó limpia otra vez.

In [10]:
%%sql
-- Celda 1.6
SELECT count(*) AS documentos,
       sum(CASE WHEN abs(monto_iva - round(monto_neto * 0.19)) > 2
                THEN 1 ELSE 0 END) AS errores_iva,
       sum(CASE WHEN abs(monto_total - (monto_neto + monto_exento + monto_iva)) > 2
                THEN 1 ELSE 0 END) AS errores_suma,
       sum(CASE WHEN monto_total < 0 AND tipo_dte <> 61
                THEN 1 ELSE 0 END) AS errores_negativos
FROM recepcion_lab06

documentos,errores_iva,errores_suma,errores_negativos
1000,0,0,0


## Paso 2 — Crear la rama

Mil documentos y cero descuadres. La tabla volvió a estar bien.

Pero fíjate en lo que pasó: **los datos malos estuvieron publicados**. Poco rato, pero
lo estuvieron. En una administración tributaria, "poco rato" puede ser el rato en que
alguien sacó un reporte.

Hagámoslo bien. Primero, una **rama**: una libreta paralela donde trabajar sin que
nadie lo vea.

In [11]:
%%sql
-- Celda 2.1
ALTER TABLE recepcion_lab06 CREATE BRANCH revision_junio

Veamos qué tiene la tabla ahora. `.refs` lista las ramas y etiquetas: `main` es la
oficial, la que cuelga del clavo.

In [12]:
%%sql
-- Celda 2.2
SELECT name, type, snapshot_id
FROM mi_espacio.recepcion_lab06.refs
ORDER BY type, name

name,type,snapshot_id
main,BRANCH,3131769625158961713
revision_junio,BRANCH,3131769625158961713


## Paso 3 — Escribir en la rama

Dos ramas, las dos apuntando a la misma página: la rama nueva nace siendo una
copia exacta de la oficial. Todavía no cambió nada.

Ahora carguemos **el mismo lote sospechoso**, pero dirigido a la rama. Fíjate en el
nombre de la tabla: tu espacio, tu tabla, y `branch_` seguido del nombre de la rama.

In [13]:
%%sql
-- Celda 3.1
INSERT INTO mi_espacio.recepcion_lab06.branch_revision_junio
SELECT * FROM curso.recepcion_sospechosa

Cargó. Veamos la rama: para leerla se pide la tabla `VERSION AS OF` con el
**nombre de la rama entre comillas**, igual que pedías una versión por identificador
en el laboratorio 01.

In [14]:
%%sql
-- Celda 3.2
SELECT count(*)         AS documentos,
       sum(monto_total) AS total_del_mes
FROM recepcion_lab06 VERSION AS OF 'revision_junio'

documentos,total_del_mes
1500,13693496696.68


Mil quinientos documentos en la rama. Y ahora **la misma consulta sobre la tabla
oficial**, sin pedir ninguna versión.

In [15]:
%%sql
-- Celda 3.3
SELECT count(*)         AS documentos,
       sum(monto_total) AS total_del_mes
FROM recepcion_lab06

documentos,total_del_mes
1000,9117709433.18


## Paso 4 — Auditar

**Mil.** Los mismos de siempre, y el mismo total.

Compara las dos celdas anteriores, porque en eso consiste todo el laboratorio: el lote
está cargado y se puede consultar, **y los analistas de la DGT no ven nada**. Para
ellos la tabla no ha cambiado.

Ahora sí, auditemos. Los mismos tres controles del paso 1, esta vez **sobre la
rama**.

In [16]:
%%sql
-- Celda 4.1
SELECT count(*) AS documentos,
       sum(CASE WHEN abs(monto_iva - round(monto_neto * 0.19)) > 2
                THEN 1 ELSE 0 END) AS errores_iva,
       sum(CASE WHEN abs(monto_total - (monto_neto + monto_exento + monto_iva)) > 2
                THEN 1 ELSE 0 END) AS errores_suma,
       sum(CASE WHEN monto_total < 0 AND tipo_dte <> 61
                THEN 1 ELSE 0 END) AS errores_negativos
FROM recepcion_lab06 VERSION AS OF 'revision_junio'

documentos,errores_iva,errores_suma,errores_negativos
1500,13,20,7


Los mismos descuadres de antes, con una diferencia enorme: **están en la rama, no
en la tabla**.

El veredicto.

In [17]:
%%sql
-- Celda 4.2
SELECT count(*) AS documentos_que_no_cuadran
FROM recepcion_lab06 VERSION AS OF 'revision_junio'
WHERE abs(monto_iva - round(monto_neto * 0.19)) > 2
   OR abs(monto_total - (monto_neto + monto_exento + monto_iva)) > 2
   OR (monto_total < 0 AND tipo_dte <> 61)

documentos_que_no_cuadran
20


## Paso 5 — Decidir: descartar

Veinte documentos no cuadran. **El lote no pasa.**

Y aquí es donde tienes algo que antes no tenías: **puedes decidir**. Hay dos caminos y
vamos a recorrer los dos.

El primero es el más simple: **descartar**. Se bota la rama entera y no pasó nada.

In [18]:
%%sql
-- Celda 5.1
ALTER TABLE recepcion_lab06 DROP BRANCH revision_junio

¿Y la tabla oficial?

In [19]:
%%sql
-- Celda 5.2
SELECT count(*) AS documentos,
       sum(CASE WHEN abs(monto_iva - round(monto_neto * 0.19)) > 2
                THEN 1 ELSE 0 END) AS errores_iva,
       sum(CASE WHEN abs(monto_total - (monto_neto + monto_exento + monto_iva)) > 2
                THEN 1 ELSE 0 END) AS errores_suma,
       sum(CASE WHEN monto_total < 0 AND tipo_dte <> 61
                THEN 1 ELSE 0 END) AS errores_negativos
FROM recepcion_lab06

documentos,errores_iva,errores_suma,errores_negativos
1000,0,0,0


Intacta: mil documentos, cero descuadres. **Nadie vio nunca esos datos.**

Y la rama ya no existe.

In [20]:
%%sql
-- Celda 5.3
SELECT name, type
FROM mi_espacio.recepcion_lab06.refs
ORDER BY type, name

name,type
main,BRANCH


## Paso 5 (continúa) — Decidir: corregir y publicar

Solo queda `main`.

Pero descartar el lote entero no siempre es lo que se quiere: veinte documentos malos
de mil quinientos no son razón para botar los otros mil cuatrocientos ochenta. El
segundo camino es **sanear y publicar**.

Rama nueva.

In [21]:
%%sql
-- Celda 5.4
ALTER TABLE recepcion_lab06 CREATE BRANCH revision_junio_v2

Y ahora cargamos el mismo lote **filtrando** los documentos que no cuadran. El
`WHERE` es el mismo control de antes, al revés: en vez de buscar los malos, deja pasar
los buenos.

In [22]:
%%sql
-- Celda 5.5
INSERT INTO mi_espacio.recepcion_lab06.branch_revision_junio_v2
SELECT * FROM curso.recepcion_sospechosa
WHERE abs(monto_iva - round(monto_neto * 0.19)) <= 2
  AND abs(monto_total - (monto_neto + monto_exento + monto_iva)) <= 2
  AND (monto_total >= 0 OR tipo_dte = 61)

Auditemos la rama nueva, con el control de siempre.

In [23]:
%%sql
-- Celda 5.6
SELECT count(*) AS documentos,
       sum(CASE WHEN abs(monto_iva - round(monto_neto * 0.19)) > 2
                THEN 1 ELSE 0 END) AS errores_iva,
       sum(CASE WHEN abs(monto_total - (monto_neto + monto_exento + monto_iva)) > 2
                THEN 1 ELSE 0 END) AS errores_suma,
       sum(CASE WHEN monto_total < 0 AND tipo_dte <> 61
                THEN 1 ELSE 0 END) AS errores_negativos
FROM recepcion_lab06 VERSION AS OF 'revision_junio_v2'

documentos,errores_iva,errores_suma,errores_negativos
1480,0,0,0


**Mil cuatrocientos ochenta documentos y cero descuadres.** Este lote sí pasa.

Para publicarlo hay que decirle a la rama oficial que apunte a donde apunta la rama de
revisión. Y para eso necesitas el identificador de esa página, que sale de `.refs`.

In [24]:
%%sql
-- Celda 5.7
SELECT snapshot_id
FROM mi_espacio.recepcion_lab06.refs
WHERE name = 'revision_junio_v2'

snapshot_id
8775591242392828658


Ese es el número. **Cópialo** y publiquemos: `REPLACE BRANCH main` mueve la rama
oficial a esa página.

Escribe la sentencia terminándola con el número que copiaste en la celda anterior,
sin comillas.

In [25]:
%%sql
-- Celda 5.8
ALTER TABLE recepcion_lab06 REPLACE BRANCH main AS OF VERSION <TU_SNAPSHOT_DE_LA_RAMA>

Publicado. Ahora la tabla oficial, la que ven los analistas.

In [26]:
%%sql
-- Celda 5.9
SELECT count(*) AS documentos,
       sum(CASE WHEN abs(monto_iva - round(monto_neto * 0.19)) > 2
                THEN 1 ELSE 0 END) AS errores_iva,
       sum(CASE WHEN abs(monto_total - (monto_neto + monto_exento + monto_iva)) > 2
                THEN 1 ELSE 0 END) AS errores_suma,
       sum(CASE WHEN monto_total < 0 AND tipo_dte <> 61
                THEN 1 ELSE 0 END) AS errores_negativos
FROM recepcion_lab06

documentos,errores_iva,errores_suma,errores_negativos
1480,0,0,0


**Mil cuatrocientos ochenta documentos, cero descuadres**, en la tabla que
consulta todo el mundo. Y en ningún momento estuvo publicado nada malo.

La rama de revisión ya cumplió su función: lo que tenía está publicado. Se bota, igual
que la anterior. Dejar ramas viejas dando vueltas es como dejar libretas a medio
llenar sobre el mostrador.

In [27]:
%%sql
-- Celda 5.10
ALTER TABLE recepcion_lab06 DROP BRANCH revision_junio_v2

## Paso 6 — Etiquetar el cierre

Falta una cosa. Este es el estado con el que la DGT cierra el mes, y dentro de
seis meses alguien va a preguntar por él. Podrías anotarte el identificador de la
página… o ponerle un nombre.

In [28]:
%%sql
-- Celda 6.1
ALTER TABLE recepcion_lab06 CREATE TAG cierre_junio_2026

Ahora se puede pedir la tabla **por ese nombre**, igual que pedías una rama.

In [29]:
%%sql
-- Celda 6.2
SELECT count(*)         AS documentos,
       sum(monto_total) AS total_del_mes
FROM recepcion_lab06 VERSION AS OF 'cierre_junio_2026'

documentos,total_del_mes
1480,13576071743.67


Ese es el cierre de junio, y lo va a seguir siendo dentro de seis meses aunque la
tabla haya cambiado veinte veces.

Miremos por última vez qué tiene la tabla.

In [30]:
%%sql
-- Celda 6.3
SELECT name, type, snapshot_id
FROM mi_espacio.recepcion_lab06.refs
ORDER BY type, name

name,type,snapshot_id
main,BRANCH,8775591242392828658
cierre_junio_2026,TAG,8775591242392828658


---

## Listo

Una rama —`main`, la oficial— y una etiqueta con nombre propio.

### Lo que te llevas

| Lo que necesitas | La sentencia |
|---|---|
| Abrir una libreta paralela | `ALTER TABLE t CREATE BRANCH nombre` |
| Escribir en ella | `INSERT INTO <db>.<tabla>.branch_nombre SELECT …` |
| Leerla | `SELECT … FROM t VERSION AS OF 'nombre'` |
| Botarla | `ALTER TABLE t DROP BRANCH nombre` |
| Publicarla | `ALTER TABLE t REPLACE BRANCH main AS OF VERSION <id>` |
| Marcar un estado | `ALTER TABLE t CREATE TAG nombre` |
| Ver ramas y etiquetas | `SELECT … FROM <db>.<tabla>.refs` |

Y el cambio de fondo, que no es una sentencia: **entre cargar y publicar hay ahora un
espacio donde revisar.** Antes de este laboratorio, cargar era publicar.

Para la DGT eso significa que un lote con errores deja de ser un problema de
credibilidad y pasa a ser un trámite: se audita en la rama, se descarta o se sanea, y a
la tabla oficial llega solo lo que cuadra.

### Dos cosas de esta versión que conviene saber

- **`CALL system.fast_forward` no existe todavía** en la versión de Iceberg de tu
  plataforma. En versiones nuevas publica una rama sin que tengas que copiar ningún
  identificador. Aquí se hace con `REPLACE BRANCH main AS OF VERSION`, que es lo que
  usaste, y funciona igual de bien.
- **`AS OF BRANCH 'nombre'` tampoco existe**: al publicar hay que dar el identificador
  de la página, no el nombre de la rama. Por eso lo sacaste de `.refs`.

### La pregunta del laboratorio siguiente

Todo lo que hiciste en el curso fue sobre tablas Iceberg que creaste tú, desde cero.

Pero la DGT no parte de cero. Tiene **años de documentos en tablas Hive**, las que ya
están en producción y de las que dependen los reportes de todos los meses.

**¿Cómo se pasan a Iceberg?** ¿Hay que copiarlo todo? ¿Se puede hacer sin apagar
nada?

Eso es el **Laboratorio 07 — Mudarse sin cerrar el negocio**.
